# 04. Tree BFS Patterns and Views

BFS processes a tree **level by level** with a queue. Use it whenever the question mentions **levels, width, views, or nearest to the root**.

**Learning order:** TEMPLATE → LEVEL AVERAGES → BOTTOM-UP → VERTICAL ORDER → MAXIMUM WIDTH → INTERVIEW THINKING

## 1. Template

```python
q = deque([root])
while q:
    for _ in range(len(q)):      # exactly one level
        node = q.popleft()
        ...                      # process the node
        if node.left:  q.append(node.left)
        if node.right: q.append(node.right)
```

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree_from_list(vals):             # level order, None = missing child
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q, i = deque([root]), 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root

def level_order(root):
    if not root:
        return []
    q, res = deque([root]), []
    while q:
        res.append([])
        for _ in range(len(q)):
            node = q.popleft()
            res[-1].append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
    return res

def inorder(root):
    return inorder(root.left) + [root.val] + inorder(root.right) if root else []

def average_of_levels(root):
    q, res = deque([root]), []
    while q:
        size, total = len(q), 0
        for _ in range(size):
            node = q.popleft()
            total += node.val
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        res.append(total / size)
    return res

t = tree_from_list([3, 9, 20, None, None, 15, 7])
print(average_of_levels(t))          # [3.0, 14.5, 11.0]
print(level_order(t)[::-1])          # [[15, 7], [9, 20], [3]]  (bottom-up = reverse of level order)

[3.0, 14.5, 11.0]
[[15, 7], [9, 20], [3]]


## 2. Vertical Order Traversal

Give each node a **column**: root = 0, left child = `col - 1`, right child = `col + 1`. Group nodes by column, then read columns from left to right. BFS keeps top-to-bottom order inside a column.

Real life: the view of a tree from above, columns from left to right.

In [2]:
from collections import defaultdict

def vertical_order(root):
    if not root:
        return []
    cols = defaultdict(list)
    q = deque([(root, 0)])
    while q:
        node, col = q.popleft()
        cols[col].append(node.val)
        if node.left: q.append((node.left, col - 1))
        if node.right: q.append((node.right, col + 1))
    return [cols[c] for c in sorted(cols)]

print(vertical_order(tree_from_list([3, 9, 20, None, None, 15, 7])))   # [[9], [3, 15], [20], [7]]

[[9], [3, 15], [20], [7]]


## 3. Maximum Width of a Binary Tree

Width of a level = distance between the leftmost and rightmost **non-null** nodes, counting the gaps. Number nodes like a heap: left child `2i`, right child `2i + 1`. Width = `last index - first index + 1`.

**Trick:** subtract the first index of each level so numbers do not grow huge in deep trees (Python handles big ints, but other languages overflow).

In [3]:
def width_of_binary_tree(root):
    q, best = deque([(root, 0)]), 0
    while q:
        first = q[0][1]
        last = first
        for _ in range(len(q)):
            node, idx = q.popleft()
            idx -= first
            last = idx
            if node.left: q.append((node.left, 2 * idx))
            if node.right: q.append((node.right, 2 * idx + 1))
        best = max(best, last + 1)
    return best

print(width_of_binary_tree(tree_from_list([1, 3, 2, 5, 3, None, 9])))   # 4

4


## 4. Interview thinking

**Signals:** "level", "layer", "view from right/left/top", "minimum depth", "width", "connect nodes at the same level".

**Say:** "BFS with a queue and a level-size loop. O(n) time, O(w) space where `w` is the maximum width."

**DFS vs BFS:** DFS is simpler for subtree computations. BFS is natural for anything about levels. A DFS with a `depth` parameter can also build level lists (`res[depth].append(val)`).

**Common mistakes:** no level-size loop, using `list.pop(0)`, ignoring the gaps when computing width.

## Quick Revision

- BFS = queue + level-size loop.
- Averages / views / bottom-up are small tweaks of level order.
- Vertical order: column index, group in a dict.
- Width: heap-style indices `2i`, `2i+1`, normalize per level.